# Análisis mensual y construcción de variables climáticas intra-anuales basadas en pronósticos

Este notebook integra toma como referencia las variables climáticas pronosticadas para los años siguientes, a fin de generar la selección adecuada de índices para el modelo en los años siguientes


In [1]:
# Librerías
from pathlib import Path
import numpy as np
import pandas as pd

In [2]:
# Raíz del proyecto y rutas
PROJECT_ROOT = Path.cwd()

while not (PROJECT_ROOT / "data").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

PATH_PROCESSED = PROJECT_ROOT / "data" / "processed"
PATH_MODEL = PROJECT_ROOT / "data" / "model"

In [3]:
# Carga de datos
clima_mes = pd.read_csv(PATH_PROCESSED / "clima_mes_2005-2027.csv")
clima_mes["date"] = pd.to_datetime(clima_mes["date"])

clima_hist = clima_mes[clima_mes["anio"] <= 2024].copy()
clima_fut = clima_mes[clima_mes["anio"] >= 2025].copy()

In [4]:
vars_base = [
    "precip_mm",
    "temp_mean",
    "temp_max",
    "et_real_mm",
    "et_potencial_mm",
    "ndvi_mean"
]

climatologia = (
    clima_hist
    .groupby(["municipio", "mes"])[vars_base]
    .agg(["mean", "std"])
)

climatologia.columns = [
    f"{var}_clim_{stat}" for var, stat in climatologia.columns
]

climatologia = climatologia.reset_index()

In [5]:
# Consolidación de datos climatológicos
clima_fut = clima_fut.merge(
    climatologia,
    on=["municipio", "mes"],
    how="left"
)

In [6]:
for v in vars_base:
    media_col = f"{v}_clim_mean"
    std_col = f"{v}_clim_std"
    
    clima_fut[f"{v}_z"] = (
        (clima_fut[v] - clima_fut[media_col])
        / clima_fut[std_col]
    )

    # Evitar infinitos por std = 0
    clima_fut[f"{v}_z"] = clima_fut[f"{v}_z"].replace([np.inf, -np.inf], np.nan)

In [ ]:
# Cálculo de variables hidrológicas
clima_fut["balance_hidrico_mm"] = clima_fut["precip_mm"] - clima_fut["et_real_mm"]
clima_fut["deficit_hidrico_mm"] = (clima_fut["et_potencial_mm"] - clima_fut["precip_mm"]).clip(lower=0)
clima_fut["exceso_hidrico_mm"] = (clima_fut["precip_mm"] - clima_fut["et_potencial_mm"]).clip(lower=0)

In [8]:
# Agregación anual por municipio
features_fut = (
    clima_fut
    .groupby(["municipio", "anio"])
    .agg({
        "precip_mm": ["sum", "mean", "std"],
        "temp_mean": ["mean", "std"],
        "ndvi_mean": ["mean", "min"],
        "et_real_mm": ["sum"],
        "balance_hidrico_mm": ["mean", "min"]
    })
)

In [9]:
# Selección de features
features_fut.columns = ["_".join(c) for c in features_fut.columns]
features_fut = features_fut.reset_index()

In [ ]:
# Guardado de resultados

ruta_out = PATH_MODEL / "features_intra_anuales_2025-2027.csv"
features_fut.to_csv(ruta_out, index=False)
print("Features futuras generadas:", ruta_out)

✔ Features futuras generadas: /home/ddayann/proyectos/Coffe/proyecto_aplicado_en_analitica_de_datos/data/model/features_intra_anuales_2025-2027.csv
